# Notebook 1: Descarga y Exploración Inicial de Datos

**DataJam Edición 4 — Universidad Distrital Francisco José de Caldas**

## Objetivo
Descargar los datasets desde el Portal de Datos Abiertos de Bogotá y realizar una exploración inicial para entender su estructura, calidad y cobertura.

## Pregunta de investigación
> ¿Existe una relación significativa entre las condiciones socioeconómicas de las localidades y la tasa de deserción escolar en colegios oficiales de Bogotá?

## Fuentes de datos
| Dataset | Fuente | Formato |
|---|---|---|
| Pobreza y Desigualdad | Sec. Distrital de Salud / DANE | CSV |
| Tasa de Deserción por UPL | Sec. Distrital de Educación | GeoJSON |
| Vulnerabilidad Calidad del Agua | Sec. Distrital de Ambiente | GeoJSON |
| Violencia Intrafamiliar | Sec. Distrital de Salud | CSV |
| Matrícula Jornada Única | Sec. Distrital de Educación | GeoJSON |
| Encuesta Multipropósito 2021 | SDP / DANE | CSV |
| Encuesta Distrital Percepción 2025 | SDP | CSV |

## 1.1 Descarga de datos

Ejecutamos el script de descarga que trae los datos directamente del portal de datos abiertos.

In [1]:
import subprocess
import sys
import os
from pathlib import Path

# El notebook está en notebooks/, subimos un nivel para llegar a la raíz del proyecto
# Usamos __vsc_ipynb_file__ si está disponible (VS Code/Kiro), sino resolvemos desde cwd
if '__vsc_ipynb_file__' in dir():
    PROJECT_ROOT = Path(__vsc_ipynb_file__).resolve().parent.parent
else:
    # Fallback: buscar requirements.txt subiendo desde cwd
    PROJECT_ROOT = Path(os.path.abspath('.')).resolve()
    for _ in range(10):
        if (PROJECT_ROOT / 'requirements.txt').exists() and (PROJECT_ROOT / 'scripts').exists():
            break
        PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f"Directorio del proyecto: {PROJECT_ROOT}")

# Descargar datos (sin los opcionales grandes)
result = subprocess.run(
    [sys.executable, 'scripts/descargar_datos.py', '--sin-opcionales'],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print("ERRORES:", result.stderr)

Directorio del proyecto: /home/aletarget/Documents/DataJam/ProblemasEconomicosYRelacionconperformanceenestudiantes/DataJam
╔══════════════════════════════════════════════════════════════╗
║  DESCARGA DE DATOS — DataJam Edición 4                      ║
║  Portal de Datos Abiertos de Bogotá                         ║
╚══════════════════════════════════════════════════════════════╝

  Directorio de datos: /home/aletarget/Documents/DataJam/ProblemasEconomicosYRelacionconperformanceenestudiantes/DataJam/data

════════════════════════════════════════════════════════════
  Pobreza y Desigualdad en Bogotá D.C.
  Fuente: Secretaría Distrital de Salud / OSB | Licencia: CC-BY-4.0
════════════════════════════════════════════════════════════
  ✓ Ya existe: osb_demografia-pobrezaygini.csv (28.6 KB)
  ✓ Ya existe: metadato_osb_demografia-pobrezaygini.csv (1.5 KB)

════════════════════════════════════════════════════════════
  Tasa de Deserción por UPL
  Fuente: Secretaría Distrital de Educación | Lice

## 1.2 Verificación de datos descargados

In [2]:
from pathlib import Path

DATA_DIR = PROJECT_ROOT / 'data'

print("Archivos descargados:")
print("=" * 60)
for p in sorted(DATA_DIR.rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  {p.relative_to(DATA_DIR)}  ({size_mb:.2f} MB)")

Archivos descargados:
  desercion_upl/tasas_upl.geojson  (2.43 MB)
  encuesta_distrital/.gitkeep  (0.00 MB)
  encuesta_distrital/20260331_diccionario_base_ano_movil_2025.xlsx  (0.08 MB)
  encuesta_distrital/base_ano_movil_2025.csv  (9.32 MB)
  encuesta_multiproposito/em2021.csv  (1196.76 MB)
  encuesta_multiproposito/em2021_inasistencia_escolar.csv  (1.62 MB)
  encuesta_multiproposito/em2021_resumen_localidad.csv  (0.00 MB)
  encuesta_multiproposito/em2021_tiempo_colegio.csv  (0.22 MB)
  matricula/matriculaciones.geojson  (0.22 MB)
  pobreza/metadato_osb_demografia-pobrezaygini.csv  (0.00 MB)
  pobreza/osb_demografia-pobrezaygini.csv  (0.03 MB)
  violencia_intrafamiliar/osb_saludmental-vintrafamiliar.csv  (105.51 MB)
  vulnerabilidad_agua/vulnerabilidad_agua.geojson  (2.39 MB)


## 1.3 Exploración: Pobreza y Desigualdad

In [3]:
import pandas as pd
import numpy as np

df_pob = pd.read_csv(DATA_DIR / 'pobreza' / 'osb_demografia-pobrezaygini.csv',
                     sep=';', encoding='latin-1')
df_pob.columns = ['Año', 'Localidad', 'Indicador', 'Categoría', 'Sexo', 'Valor']
df_pob['Valor'] = df_pob['Valor'].astype(str).str.replace(',', '.').astype(float)
df_pob['Año'] = df_pob['Año'].astype(int)

print(f"Shape: {df_pob.shape}")
print(f"\nAños disponibles: {sorted(df_pob['Año'].unique())}")
print(f"\nIndicadores: {df_pob['Indicador'].unique().tolist()}")
print(f"\nLocalidades: {df_pob['Localidad'].nunique()} únicas")
df_pob.head(10)

Shape: (447, 6)

Años disponibles: [np.int64(2003), np.int64(2011), np.int64(2014), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Indicadores: ['Privaciones', 'IPM', 'Contrib IPM', 'Coeficiente de Gini', 'Pobreza monetaria', 'Pobreza monetaria extrema']

Localidades: 21 únicas


,Año,Localidad,Indicador,Categoría,Sexo,Valor
0,2018,Bogotá D.C.,Privaciones,Analfabetismo,Ambos sexos,1.6
1,2018,Bogotá D.C.,Privaciones,Bajo logro educativo,Ambos sexos,18.6
2,2018,Bogotá D.C.,Privaciones,Barreras a servicios para cuidado de la primer...,Ambos sexos,9.4
3,2018,Bogotá D.C.,Privaciones,Barreras de acceso a servicios de salud,Ambos sexos,1.2
4,2018,Bogotá D.C.,Privaciones,Desempleo de larga duración,Ambos sexos,12.8
5,2018,Bogotá D.C.,Privaciones,Hacinamiento crítico,Ambos sexos,5.3
6,2018,Bogotá D.C.,Privaciones,Inadecuada eliminación de excretas,Ambos sexos,0.4
7,2018,Bogotá D.C.,Privaciones,Inasistencia escolar,Ambos sexos,1.4
8,2018,Bogotá D.C.,Privaciones,Material inadecuado de paredes exteriores,Ambos sexos,0.2
9,2018,Bogotá D.C.,Privaciones,Material inadecuado de pisos,Ambos sexos,0.0


In [4]:
# Pobreza monetaria por localidad (último año disponible)
ultimo_ano = df_pob['Año'].max()
pobreza_mon = df_pob[
    (df_pob['Indicador'] == 'Pobreza monetaria') &
    (df_pob['Año'] == ultimo_ano) &
    (df_pob['Sexo'].str.contains('Ambos', na=False)) &
    (~df_pob['Localidad'].str.contains('Bogot', na=False))
].sort_values('Valor', ascending=False)

print(f"Pobreza monetaria por localidad ({ultimo_ano}):")
print("=" * 40)
for _, row in pobreza_mon.iterrows():
    print(f"  {row['Localidad']:25s} {row['Valor']:.1f}%")

Pobreza monetaria por localidad (2025):


## 1.4 Exploración: Tasa de Deserción por UPL

In [5]:
import json

with open(DATA_DIR / 'desercion_upl' / 'tasas_upl.geojson') as f:
    gj = json.load(f)

print(f"Tipo: {gj['type']}")
print(f"Número de features: {len(gj['features'])}")
print(f"\nPropiedades del primer feature:")
print(json.dumps(gj['features'][0]['properties'], indent=2, default=str)[:800])

Tipo: FeatureCollection
Número de features: 33

Propiedades del primer feature:
{
  "OBJECTID": 1,
  "CODIGO_UPL": "UPL13",
  "TtotalAprNOf_UPL": 97.38810868,
  "TtotalAprOf_UPL": 89.87929434,
  "TtotalDeserNOf_UPL": 2.113138207,
  "TtotalDeserOf_UPL": 2.441422251,
  "TtotalReprNOf_UPL": 0.498753117,
  "TtotalReprOf_UPL": 7.679283413,
  "Fecha": "2024-12-31T00:00:00Z",
  "SHAPE_Length": 17186.318530654695,
  "SHAPE_Area": 13010424.388147624,
  "NOM_UPL": "13"
}


In [6]:
# Convertir a DataFrame
deser_rows = []
for feat in gj['features']:
    p = feat['properties']
    cod_upl = p.get('CODIGO_UPL') or p.get('codigo_upl', '')
    nom_upl = p.get('NOM_UPL') or p.get('nom_upl', '')
    deser_rows.append({
        'Cod_UPL': cod_upl,
        'NOM_UPL': nom_upl,
        'Desercion_Of': p.get('TtotalDeserOf_UPL') or p.get('ttotal_deser_of_upl', 0),
        'Reprobacion_Of': p.get('TtotalReprOf_UPL') or p.get('ttotal_repr_of_upl', 0),
        'Aprobacion_Of': p.get('TtotalAprOf_UPL') or p.get('ttotal_apr_of_upl', 0),
    })

df_deser = pd.DataFrame(deser_rows)
print(f"UPLs con datos de deserción: {len(df_deser)}")
print(f"\nEstadísticas descriptivas:")
df_deser[['Desercion_Of', 'Reprobacion_Of', 'Aprobacion_Of']].describe()

UPLs con datos de deserción: 33

Estadísticas descriptivas:


,Desercion_Of,Reprobacion_Of,Aprobacion_Of
count,33.000000,33.000000,33.000000
mean,2.721831,7.706997,89.571172
std,0.764620,1.940450,2.415110
min,1.570048,5.027577,82.385422
25%,2.199989,6.417654,89.097882
50%,2.479236,7.371148,90.082160
75%,3.101675,8.385867,91.350559
max,4.681873,13.583655,92.576060


## 1.5 Exploración: Violencia Intrafamiliar

In [7]:
# Este archivo es grande (~110MB), cargamos solo las columnas necesarias
vif_path = DATA_DIR / 'violencia_intrafamiliar' / 'osb_saludmental-vintrafamiliar.csv'
if vif_path.exists():
    df_vif = pd.read_csv(vif_path, sep=';', encoding='utf-8-sig',
                         usecols=['ano', 'grupoedad', 'NOMBRE_LOCALIDAD'],
                         low_memory=False)
    print(f"Shape: {df_vif.shape}")
    print(f"\nAños: {sorted(df_vif['ano'].unique())}")
    print(f"\nGrupos de edad: {df_vif['grupoedad'].unique().tolist()}")
    print(f"\nCasos en menores (2020-2025):")
    menores = df_vif[
        (df_vif['grupoedad'].isin(['De 1 - 5 años', 'De 6 - 13 años', 'De 14 - 17 años'])) &
        (df_vif['ano'].between(2020, 2025))
    ]
    print(f"  Total: {len(menores):,} casos")
    print(f"  Por localidad (top 5):")
    print(menores['NOMBRE_LOCALIDAD'].value_counts().head())
else:
    print("⚠ Archivo no descargado aún.")

Shape: (492837, 3)

Años: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

Grupos de edad: ['De 1 - 5 años', 'De 14 - 17 años', 'De 18 - 26 años', 'De 27 - 44 años', 'De 45 - 59 años', 'De 6 - 13 años', 'De 60 - 69 años', 'De 70 - 79 años', 'De 80 - 99 años', 'Menor 1 año', 'De 100 - 105 años', '14 - 17 años', 'De 30 - 34 años']

Casos en menores (2020-2025):
  Total: 148,929 casos
  Por localidad (top 5):
NOMBRE_LOCALIDAD
Ciudad Bolívar    22685
Kennedy           20712
Bosa              19209
Suba              17037
Usme              12734
Name: count, dtype: int64


## 1.6 Exploración: Matrícula

In [8]:
mat_path = DATA_DIR / 'matricula' / 'matriculaciones.geojson'
if mat_path.exists():
    with open(mat_path) as f:
        gj_mat = json.load(f)
    print(f"Features (colegios/sedes): {len(gj_mat['features'])}")
    print(f"\nPropiedades disponibles:")
    print(list(gj_mat['features'][0]['properties'].keys())[:15])
else:
    print("⚠ Archivo no descargado aún.")

Features (colegios/sedes): 379

Propiedades disponibles:
['NOMBRE_EST', 'NOMBRE_SED', 'DIRECCION', 'SECTOR', 'NATU_JUR', 'ESTADO', 'CALENDARIO', 'GENERO', 'CARACTER_P', 'ESPECIALID', 'CLASE_TIPO', 'BILINGUE', 'FECHA', 'DANE12_EST', 'DANE12_SED']


## 1.7 Resumen de calidad de datos

| Dataset | Registros | Cobertura temporal | Unidad geográfica | Valores nulos |
|---|---|---|---|---|
| Pobreza | ~X | 2011-2025 | Localidad | Revisar |
| Deserción UPL | 33 UPLs | 2024 | UPL | Mínimos |
| Violencia | ~X | 2012-2025 | Localidad | Revisar |
| Matrícula | ~X colegios | 2025 | Colegio/Localidad | Mínimos |

### Conclusiones de la exploración
1. Los datos de pobreza están a nivel de **localidad** y los de deserción a nivel de **UPL**. Necesitamos un mapeo UPL→Localidad.
2. La Encuesta Multipropósito tiene microdatos que permiten analizar transporte y economía a nivel individual.
3. El dataset de violencia intrafamiliar es grande pero filtrable por grupo de edad.

### Siguiente paso
→ Notebook 02: Limpieza, integración y mapeo territorial.